<center>
<h1>
<h1>APM 53674: ALTeGraD</h1>
<h2>Lab Session 4: Distillation and Retrieval Augmented Generation</h2>
<h4>Lecture: Dr. Guokan Shang<br>
Lab: Yang Zhang and Xiao Fei</h4>
<h5>Tuesday, November 18, 2025</h5>
<br>
</center>

<hr style="border:10px solid gray"> </hr>
<p style="text-align: justify;">
This handout includes theoretical introductions, <font color='blue'>coding tasks</font> and <font color='red'>questions</font>. Before the deadline, you should submit <a href='https://forms.gle/9dyaes6dimfvyjwq6' target="_blank">here</a> a <B>.ipynb</B> file named <b>Lastname_Firstname.ipynb</b> containing your notebook (with the gaps filled and your answers to the questions). Your answers should be well constructed and well justified. They should not repeat the question or generalities in the handout. When relevant, you are welcome to include figures, equations and tables derived from your own computations, theoretical proofs or qualitative explanations. One submission is required for each student. The deadline for this lab is <b>Novemver 23
, 2025 11:59 PM</b>. No extension will be granted. Late policy is as follows: ]0, 24] hours late → -5 pts; ]24, 48] hours late → -10 pts; > 48 hours late → not graded (zero).
</p>
<hr style="border:5px solid gray"> </hr>

# Install Requirements

In [ ]:
# Install required dependencies
!pip -q install torch tqdm jsonlines h5py
!pip -q install --upgrade transformers accelerate vllm
!pip install jedi
# !pip -q install datasets==2.21.0 pandas==2.2.2
# !pip -q install chromadb==0.4.22
# !pip -q install "numpy<2.0" --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.9/312.9 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/4

# Part 1 - Model Distillation

> In this section, you’ll learn the difference between **white-box** and **black-box** distillation, generate **synthetic data** to train a **student model**, and implement **white-box distillation** to specialize a model for a **RAG on Wikipedia** use case.


<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 1:</br>
Explain briefly the difference between black-box and white-box distillation? </br> What are the advantages and inconvenients of each approach?
<hr style="border:10px solid red"> </hr>  
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 1: </b><br>
Your answer here.
<hr style="border:10px solid green"> </hr>
</font></h4>


In black-box distillation, we can access only the teacher’s generated outputs, such as its final answers. We cannot inspect its logits, token probabilities, hidden states, or parameters. The generated answer becomes an ordinary training target, and the student is fine-tuned using supervised learning.

Advantages:

It works with proprietary models accessible only through an API;
It does not require access to the teacher’s internal architecture;
Generated input–answer pairs are relatively easy to store and reuse;
The teacher and student can use different architectures and tokenizers.

Disadvantages:

Each position provides only the sampled teacher token, not the teacher’s complete probability distribution; Information about plausible alternative tokens is lost; Generated answers may contain errors or hallucinations that the student then learns; A large amount of synthetic data may be required; API generation can be expensive.


In white-box distillation, we have access to internal teacher information—most commonly its logits or token probabilities, and sometimes its hidden representations or attention maps.

For every token position, the student learns to approximate the teacher’s full distribution.

Advantages:

It provides a richer and smoother training signal than a single generated token; The student learns the teacher’s relative preferences among alternative tokens; It is often more data-efficient and can transfer the teacher’s behavior more accurately; Additional internal representations can potentially be distilled.

Disadvantages:

It requires access to the teacher’s internal outputs, so it generally cannot be performed with a generation-only API; Storing complete vocabulary-sized distributions is extremely expensive; Running the teacher during training requires substantial GPU memory and computation; Differences between the teacher’s and student’s tokenization or architecture can complicate distillation.

<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 2:</br>
What is the main requirement for a teacher/student pair of models to perform white-box distillation?
<hr style="border:10px solid red"> </hr>  
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 2: </b><br>
Your answer here.

For the token-level white-box distillation used in this lab, the teacher and student must have a compatible output space. In practice, they should use the same tokenizer and vocabulary, with the same token-to-ID mapping.

This is necessary because the KL divergence compares corresponding probabilities.

<hr style="border:10px solid green"> </hr>
</font></h4>


## 1.1 - Synthetic Data Generation

We're going to specialize a small 0.5B parameter model to perform RAG by distilling the abilities of a 7B parameter one.

For that we'll be using `Qwen/Qwen2.5-0.5B-Instruct` as student and `Qwen/Qwen2.5-7B-Instruct-AWQ` (quantized version of `Qwen2.5-7B-Instruct`) as teacher.  

In order to perform white-box distillation on generated answers we have two choices.

1. We can perform a forward pass with the teacher, a forward pass with student on the complete sequence, and backprop difference of logprobs using KL Loss.
2. Generation of samples with the teacher, save the logprobs and perform finetuning in a second step.

<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 3:</br>
What are the computational advantages of 1. vs 2.?
<hr style="border:10px solid red"> </hr>  
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 3: </b><br>
Your answer here.

Its main advantage is that the teacher probabilities are computed when needed, so we do not have to save enormous probability tensors to disk. It also allows the training inputs to change dynamically between epochs.

However, both models must generally be loaded simultaneously. This requires a lot of GPU memory, particularly with a 7B teacher. The teacher also performs another forward pass every time a sample is revisited, making training slower and computationally expensive.

Its main advantage is that the teacher and student do not need to fit in GPU memory simultaneously. This is particularly useful in the lab’s limited Google Colab environment. The teacher is evaluated only once per sample, and the saved probabilities can be reused across several student epochs or experiments.

Its disadvantage is storage. Saving the probability of every vocabulary token at every position would require a huge amount of disk space.

<hr style="border:10px solid green"> </hr>
</font></h4>

We're going to generate a bunch of questions related to wikipedia paragraphs.
For that we need to establish a system prompt that will allow for easy extraction.

In [ ]:
system_prompt = """
You are a question generator.
The user will provide:

```json
{"title": "the title of an article", "paragraph": "a paragraph from that article"}
```

Your task:

* Generate one clear, self-contained question that can be answered using only the provided paragraph.
* The question must be **specific**, **unambiguous**, and directly tied to the paragraph’s content.
* Return the result with the question as a valid JSON** in the form:

```json
{
  "question": "your question here"
}
```

Example:
User input:

```json
{
"title": "The Moon Landing",
"paragraph": "On July 20, 1969, Neil Armstrong became the first human to set foot on the Moon, followed by Buzz Aldrin."
}
```
Assistant output:

```json
{
  "question": "Who was the first human to set foot on the Moon during the Apollo 11 mission?"
}
```
"""

<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 4:</br>
In this system prompt, we don't generate answers, only questions. Explain why it's necessary in the context of white-box distillation.
<hr style="border:10px solid red"> </hr>  
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 4: </b><br>
Your answer here.

During the first stage, we generate only questions because the answers must later be generated by the teacher while recording its token-level log-probabilities.

If we generated and stored only the final answers now, we would retain the selected tokens but lose the teacher’s complete probability distribution over alternative tokens. This would reduce the process to black-box distillation or ordinary supervised fine-tuning.

<hr style="border:10px solid green"> </hr>
</font></h4>

<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 5:</br>
For synthetic data generation we'll be using vLLM.
vLLM is an optimized llm inference engine that can improve generation speed thanks to hardware specific optimization and computational tricks such as Prefix KV Caching.
(https://docs.vllm.ai/en/latest/features/automatic_prefix_caching.html)</br></br> 1. Explain why prefix caching will be very efficient in our case? </br></br>
2. What sampling `temperature` should we use? Justify.



<hr style="border:10px solid red"> </hr>  
</font></h4>


<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 5: </b><br>
Your answer here.

All generation requests begin with the same long system prompt containing the instructions for generating a question. When the same prefix is processed repeatedly, the transformer normally recomputes its attention keys and values for that prefix for every sample.

Prefix KV caching stores these previously computed key and value tensors.

Use T = 0.2:
When T < 1, the distribution becomes sharper, so the model is more likely to select high-probability tokens. This produces questions that are:

more stable and factual;
more likely to follow the required JSON format;
directly related to the supplied paragraph;
less likely to contain creative but unsupported content.

A high temperature would provide more diversity, but it would also increase the risk of ambiguous questions, hallucinations and invalid JSON. A temperature of exactly zero would provide deterministic greedy decoding, but \(0.2\) preserves a small amount of diversity while maintaining reliability.

<hr style="border:10px solid green"> </hr>
</font></h4>

<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 1: Optimized Generation of Synthetic Questions

Complete the below code with the adequate options (prefix caching and `temperature`)

<hr style="border:10px solid blue"> </hr>
</font></h4>


In [ ]:
!pip install --no-cache-dir torchaudio==2.11.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.7 MB/s eta 0:00:00


In [ ]:
import torch
import torchaudio

print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("torchaudio:", torchaudio.__version__)

torch: 2.13.0+cu130
CUDA: 13.0
torchaudio: 2.11.0+cu130


In [ ]:
import os

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

from vllm import LLM, SamplingParams

In [ ]:
# from vllm import LLM
# from vllm import SamplingParams
# import os
# path_teacher = "Qwen/Qwen2.5-7B-Instruct-AWQ"
# llm = LLM(model=path_teacher, gpu_memory_utilization=0.9, max_model_len=5000, enable_prefix_caching=True) # To Complete
# sampling_params = SamplingParams(temperature = 0.2, max_tokens=400) # To Complete

import sys

path_teacher = "Qwen/Qwen2.5-7B-Instruct-AWQ"

notebook_stdout = sys.stdout

try:
    sys.stdout = sys.__stdout__

    llm = LLM(
        model=path_teacher,
        gpu_memory_utilization=0.9,
        max_model_len=5000,
        enable_prefix_caching=True
    )
finally:
    sys.stdout = notebook_stdout

sampling_params = SamplingParams(
    temperature=0.2,
    max_tokens=400
)

print("vLLM initialized successfully")

INFO 08-26 08:41:40 [api_utils.py:273] non-default args: {'max_model_len': 5000, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}
WARNING 08-26 08:41:41 [arg_utils.py:1678] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 08-26 08:41:41 [model.py:645] Resolved architecture: Qwen2ForCausalLM
INFO 08-26 08:41:41 [model.py:1883] Using max model len 5000


Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

INFO 08-26 08:41:42 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
INFO 08-26 08:41:46 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-7B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=5000, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=auto_awq, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='

ValueError: Free memory on device cuda:0 (1.88/14.56 GiB) on startup is less than desired GPU memory utilization (0.9, 13.11 GiB). Decrease GPU memory utilization or reduce GPU memory used by other processes.

<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 2: Question Generation</br>

<hr style="border:10px solid blue"> </hr>
</font></h4>

We're going to use the llm.chat vLLM api to generate our samples.
Make the adequate code to generate the data and save it in a questions.jsonl file.

Entries of jsonl file should look like:
```json
{
  "id_doc": <wikipedia_article_id>,
  "id_paragraph": <paragraph_dataset_id>,
  "question": <generated_question>,
  "title": <title_wikipedia_article>,
  "paragraph": <text_of_paragraph>
}
```

You should:

1. Complete `def extract_question(generated_text: str) -> str:` to extract the generated question.
2. Complete `def conversation_generator` to output a generator directly ingestible by `llm.chat` api
3. Complete the dataloader and for loop to generate samples by batches of 4
4. Save the generated questions in a `questions.jsonl` file

In [ ]:
# # from typing import Iterable, Iterator, List
# # import json
# # def extract_question(generated_text: str) -> str:

# #     """Extract the question from the generated text. If the question is not\
# #     following the right format return None."""

# #     try:

# #       j: dict = json.loads(generated_text) # To Complete
# #       assert isinstance(j, dict)
# #       assert "question" in j
# #       assert isinstance(j["question"], str)
# #       assert len(j["question"].strip()) > 0
# #       return j["question"]

# #     except AssertionError:
# #       return None


# # def conversation_generator(
# #     entries: Iterable[dict],
# #     system_prompt: str
# #     ) -> Iterator[dict]:

# #     """Generate the conversation with the model."""

# #     for entry in entries:
# #       user_prompt = json.dumps({
# #             "title": entry["title"],
# #             "paragraph": entry["paragraph"]
# #         })
# #       conversation: List[dict] = [
# #           {"role": "system", "content": system_prompt},
# #           {"role": "user", "content": user_prompt}
# #       ]
# #       # To Complete
# #       yield conversation

# from typing import Iterable, Iterator, List
# import json


# def extract_question(generated_text: str) -> str:
#     """Extract the question from the generated text.

#     If the question does not follow the required format, return None.
#     """

#     try:
#         j: dict = json.loads(generated_text)

#         assert isinstance(j, dict)
#         assert "question" in j
#         assert isinstance(j["question"], str)
#         assert len(j["question"].strip()) > 0

#         return j["question"].strip()

#     except (json.JSONDecodeError, AssertionError, TypeError):
#         return None


# def conversation_generator(
#     entries: Iterable[dict],
#     system_prompt: str
# ) -> Iterator[List[dict]]:
#     """Generate conversations that can be passed to llm.chat."""

#     for entry in entries:
#         user_prompt = json.dumps({
#             "title": entry["title"],
#             "paragraph": entry["paragraph"]
#         })

#         conversation: List[dict] = [
#             {
#                 "role": "system",
#                 "content": system_prompt
#             },
#             {
#                 "role": "user",
#                 "content": user_prompt
#             }
#         ]

#         yield conversation

In [ ]:
from typing import Iterable, Iterator, List, Optional
import json
import re


def extract_question(generated_text: str) -> Optional[str]:
    """
    Extract the question from the model's generated text.

    Accepts:
        {"question": "..."}

    as well as Markdown-wrapped JSON:
        ```json
        {"question": "..."}
        ```

    Returns None if a valid question cannot be extracted.
    """

    try:
        # Remove leading/trailing whitespace
        text = generated_text.strip()

        # Remove Markdown code fences such as ```json ... ```
        text = re.sub(
            r"^```(?:json)?\s*",
            "",
            text,
            flags=re.IGNORECASE
        )
        text = re.sub(
            r"\s*```$",
            "",
            text
        )

        # Remove any whitespace left after removing the fences
        text = text.strip()

        # Parse JSON
        j = json.loads(text)

        # Validate expected structure
        if not isinstance(j, dict):
            return None

        if "question" not in j:
            return None

        if not isinstance(j["question"], str):
            return None

        question = j["question"].strip()

        if len(question) == 0:
            return None

        return question

    except (json.JSONDecodeError, TypeError, ValueError):
        return None


def conversation_generator(
    entries: Iterable[dict],
    system_prompt: str
) -> Iterator[List[dict]]:
    """
    Generate conversations that can be passed to llm.chat.
    """

    for entry in entries:

        user_prompt = json.dumps({
            "title": entry["title"],
            "paragraph": entry["paragraph"]
        })

        conversation: List[dict] = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]

        yield conversation

In [ ]:
from torch.utils.data import DataLoader
from datasets import load_dataset

ds = load_dataset("EvanD/Lab4_wikiparagraphs")

batch_size = 4
conversations = list(
    conversation_generator(
        entries=ds["train"],
        system_prompt=system_prompt
    )
)# To Complete

dataloader = DataLoader(conversations, batch_size=batch_size, shuffle=False, num_workers=2, prefetch_factor=2, collate_fn=lambda x: x)


In [ ]:
print("Dataset train size:", len(ds["train"]))
print("Dataloader batches:", len(dataloader))

Dataset train size: 200
Dataloader batches: 50


In [ ]:
print(ds["train"][0])
print(ds["train"][0].keys())

{'id': '39', 'title': 'Albedo', 'paragraph': 'Albedo (; ) is the fraction of sunlight that is diffusely reflected by a body. It is measured on a scale from 0 (corresponding to a black body that absorbs all incident radiation) to 1 (corresponding to a body that reflects all incident radiation).'}
dict_keys(['id', 'title', 'paragraph'])


In [ ]:
from tqdm.auto import tqdm
import jsonlines


with jsonlines.open("questions.jsonl", "w") as writer:
    for i, batch in tqdm(
        enumerate(dataloader),
        total=len(dataloader)
    ):
        id_paragraph = i * batch_size

        outputs = llm.chat(
            batch,
            sampling_params=sampling_params,
            use_tqdm=False
        )

        for j, output in enumerate(outputs):
            q = extract_question(output.outputs[0].text)

            # Verify that a valid question was extracted
            if q is not None:
                paragraph_index = id_paragraph + j
                original_entry = ds["train"][paragraph_index]

                entry = {
                    "id_doc": original_entry["id"],
                    "id_paragraph": paragraph_index,
                    "question": q,
                    "title": original_entry["title"],
                    "paragraph": original_entry["paragraph"]
                }

                writer.write(entry)

  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
import jsonlines
import os

with jsonlines.open("questions.jsonl") as reader:
    questions = list(reader)

print("Questions saved:", len(questions))
print("File size:", os.path.getsize("questions.jsonl") / 1024, "KB")
print("First entry:", questions[0] if questions else "EMPTY")

Questions saved: 200
File size: 130.4501953125 KB
First entry: {'id_doc': '39', 'id_paragraph': 0, 'question': 'What is the range of values for albedo and what do they represent?', 'title': 'Albedo', 'paragraph': 'Albedo (; ) is the fraction of sunlight that is diffusely reflected by a body. It is measured on a scale from 0 (corresponding to a black body that absorbs all incident radiation) to 1 (corresponding to a body that reflects all incident radiation).'}


In [ ]:
batch = next(iter(dataloader))

outputs = llm.chat(
    batch,
    sampling_params=sampling_params,
    use_tqdm=False
)

for i, output in enumerate(outputs):
    text = output.outputs[0].text
    print(f"\n--- OUTPUT {i} ---")
    print(repr(text))
    print("Extracted:", extract_question(text))


--- OUTPUT 0 ---
'```json\n{\n  "question": "What is the range of values for albedo and what do they represent?"\n}\n```'
Extracted: What is the range of values for albedo and what do they represent?

--- OUTPUT 1 ---
'```json\n{\n  "question": "What can the albedo for land surfaces at a specific solar zenith angle be approximated by, according to the paragraph?"\n}\n```'
Extracted: What can the albedo for land surfaces at a specific solar zenith angle be approximated by, according to the paragraph?

--- OUTPUT 2 ---
'```json\n{\n  "question": "What was the relative effect of planting new forests in high latitudes, according to studies by the Hadley Centre?"\n}\n```'
Extracted: What was the relative effect of planting new forests in high latitudes, according to studies by the Hadley Centre?

--- OUTPUT 3 ---
'```json\n{\n  "question": "What is the total sales tax rate for a meal in Mobile County, Alabama, including the restaurant tax?"\n}\n```'
Extracted: What is the total sales tax r

## 1.2 - Logprobs Generation

We get to the second part of this distillation where we are interested in distilling the answers logprobs of our teacher model to specialize our 0.5B model to perform Retrieval Augmented Generation (RAG).

First we're going to generate the logprobs with our 7B parameter model.

We'll use the following system prompt:

In [ ]:
system_prompt = (
    "You are an assistant for a Retrieval-Augmented Generation (RAG) system.\n"
    "Answer the question using only the provided documents. "
    "If the answer cannot be found in the provided documents, respond that the answer is not available in the provided document database. "
    "Documents:\n{context_block}\n\n"
)

<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 3: Complete Conversation Generator</br>

<hr style="border:10px solid blue"> </hr>
</font></h4>

Based on the previous code you made to generate questions, make a second one to generate answers and saving logprobs. We updated the sampling parameters of vLLM to return the logprobs and the token ids of the 20 most probable tokens.

As this can result in high quantity of data in practice we're going to store generated conversations in a .jsonl file and generated logprobs and token ids to a hdf5 file (see https://docs.h5py.org/en/stable/ for more information on hdf5).

`save_logprobs_hdf5()` is already implemented for you, and allows to save the logprobs to a .h5 hdf5 file, and increments sequences automatically

The structure of the conversation generator `conversation_generator()` function is implemented, you need to complete it.



In [ ]:
import h5py, os
import numpy as np

def save_logprobs_hdf5(path, sequences, start_idx=None):
    """
    sequences: list of sequences
       each sequence = list of steps
          each step = {token_id: Logprob(logprob=..., ...), ...}
    """
    mode = "a" if os.path.exists(path) else "w"
    with h5py.File(path, mode) as f:
        # choose index to start writing
        if start_idx is None:
            # auto-continue numbering if file already has data
            existing = [int(k.split("_")[1]) for k in f.keys() if k.startswith("seq_")]
            start_idx = max(existing)+1 if existing else 0

        for s_i, seq in enumerate(sequences, start=start_idx):
            g = f.create_group(f"seq_{s_i}")
            for t_i, step in enumerate(seq):
                token_ids = np.fromiter(step.keys(), dtype=np.int32)
                logprobs  = np.array([lp.logprob for lp in step.values()], dtype=np.float32)
                g.create_dataset(f"step_{t_i}/token_ids", data=token_ids, compression="gzip")
                g.create_dataset(f"step_{t_i}/logprobs",  data=logprobs,  compression="gzip")


<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 6:</br>
When completing conversation_generator we have several ways of sampling.
What are good paragraph sampling strategies we could use to ensure good performance of downstream model?

<hr style="border:10px solid red"> </hr>  
</font></h4>


<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 6: </b><br>
Your answer here.

A suitable strategy is to always include the paragraph from which the question was generated and combine it with a variable number of negative paragraphs. Some negatives should come from the same article, providing topically related and relatively difficult distractors, while others should come from unrelated articles to increase diversity. Retrieval-based hard negatives could further improve training by selecting paragraphs that are semantically similar to the question but do not contain its answer. The number and order of paragraphs should be randomized so that the model does not rely on a fixed context size or on the relevant paragraph always occupying the same position. Finally, some examples may exclude the relevant paragraph entirely, allowing the downstream model to learn when it should state that the answer is not available instead of hallucinating.

<hr style="border:10px solid green"> </hr>
</font></h4>

In [ ]:
import random

doc_id_to_paragraphs = {}

for line in ds["train"]:
    doc_id = line["id"]
    paragraph = line["paragraph"]
    title = line["title"]

    if doc_id not in doc_id_to_paragraphs:
        doc_id_to_paragraphs[doc_id] = []

    # Add the title to contextualize the paragraph
    doc_id_to_paragraphs[doc_id].append(
        title + " -- " + paragraph
    )


def conversation_generator(
    path_jsonl: str,
    system_prompt: str,
    top_k: int = 3
    ) -> Iterator[dict]:

    """Generate the conversation with the model."""

    with jsonlines.open(path_jsonl, "r") as f:
      for q_p in f:
        number_of_paragraphs_in_context = random.sample(
            range(1, top_k + 1), 1
        )[0]

        # Always include the paragraph containing the answer
        paragraphs = [
            q_p["title"] + " -- " + q_p["paragraph"]
        ]

        while len(paragraphs) < number_of_paragraphs_in_context:

          # With 50% probability, add a paragraph from the same article
          if (
              random.random() < 0.5
              and len(doc_id_to_paragraphs[q_p["id_doc"]]) > 1
          ):
            paragraph = random.choice(
                doc_id_to_paragraphs[q_p["id_doc"]]
            )

          # Add a paragraph from a different article
          else:
            other_doc_ids = [
                doc_id
                for doc_id in doc_id_to_paragraphs
                if doc_id != q_p["id_doc"]
            ]

            sampled_doc_id = random.choice(other_doc_ids)

            paragraph = random.choice(
                doc_id_to_paragraphs[sampled_doc_id]
            )

          # Avoid inserting the same paragraph twice
          if paragraph not in paragraphs:
            paragraphs.append(paragraph)

        random.shuffle(paragraphs)

        system_prompt_formatted = system_prompt.format(
            context_block="\n\n".join(
                f"[Document {i+1}]: {doc}"
                for i, doc in enumerate(paragraphs)
            )
        )

        conversation = [
            {
                "role": "system",
                "content": system_prompt_formatted
            },
            {
                "role": "user",
                "content": q_p["question"]
            }
        ]

        yield conversation

In [ ]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from typing import Iterable


batch_size = 4

conversations = list(conversation_generator(path_jsonl="questions.jsonl", system_prompt=system_prompt)) # To Complete

dataloader = DataLoader(conversations, batch_size=batch_size, shuffle=False, num_workers=2, prefetch_factor=2, collate_fn=lambda x: x)

sampling_params = SamplingParams(temperature=0.2, max_tokens=400, logprobs=20)

with jsonlines.open("conversations_rag.jsonl", "w") as writer:
    for batch in tqdm(dataloader):
        out = llm.chat(batch,
                        sampling_params=sampling_params,
                        use_tqdm=False)
        for i, b in enumerate(batch):
            text = out[i].outputs[0].text
            b.append({"role": "assistant", "content": text})
            writer.write(b)
        save_logprobs_hdf5("logprobs.h5", [out[i].outputs[0].logprobs for i in range(len(out))])

'''
conversations = list(conversation_generator(path_jsonl="questions.jsonl", system_prompt=system_prompt))
'''

  0%|          | 0/50 [00:00<?, ?it/s]

'\nconversations = list(conversation_generator(path_jsonl="questions.jsonl", system_prompt=system_prompt))\n'

In [ ]:
import jsonlines

with jsonlines.open("questions.jsonl") as reader:
    questions = list(reader)

print("Number of questions:", len(questions))
print("First question:", questions[0] if questions else "EMPTY!")

Number of questions: 200
First question: {'id_doc': '39', 'id_paragraph': 0, 'question': 'What is the range of values for albedo and what do they represent?', 'title': 'Albedo', 'paragraph': 'Albedo (; ) is the fraction of sunlight that is diffusely reflected by a body. It is measured on a scale from 0 (corresponding to a black body that absorbs all incident radiation) to 1 (corresponding to a body that reflects all incident radiation).'}


## 1.3 - KL-Divergence and Distillation

**You should restart the notebook kernel to free the gpu memory from the 7B model that is no longer needed**

Now that we have the generated conversations and their logprobs we can train our 0.5B model to output the same distribution.

The attentive student would have noticed that we have an incomplete representation of the probability distribution over tokens due to only keeping the top-20 logprobs.


<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 7:</br>
What are the two solutions you can see to approximate full distillation despite only having the top-20 logprobs?

<hr style="border:10px solid red"> </hr>  
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 7: </b><br>
Your answer here.

1- Renormalize the top-20 probabilities. Treat the top-20 tokens as the teacher’s effective distribution and normalize their probabilities so they sum to 1. Then compute the distillation loss only over those tokens. This ignores the probability mass outside the top 20 but preserves the teacher’s relative preferences among its most likely tokens.

2- Approximate the missing probability mass. Compute how much probability remains outside the top 20 and distribute or model that residual mass over the remaining vocabulary, for example using a uniform approximation or an additional “other tokens” bucket. This better approximates a full teacher distribution than simply assigning zero probability to every missing token.

<hr style="border:10px solid green"> </hr>
</font></h4>

We supply two functions to help in this implementation:

- `find_subsequence()`that allows to find all the occurences of a token subsequence in a 1-D tensor
- `get_labels()` that allows to expand the top-20 logprobs to the whole vocabulary, implicitly setting prob to zero for other tokens
- `QwenKLDataset`that loads samples from .h5 and .jsonl files, remove problematic inconsistent tokenized examples and outputs samples tokenized for training.

To simplify we will train with a batch size of 1, you can implement gradient accumulation if you wish.

The `PREFIX_ASSISTANT`variable contains the tokens that encode for

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
import torch

def find_subsequence(input_ids: torch.Tensor, subseq: torch.Tensor):
    """Find the first index of each occurence of a subsequence in the input_ids."""

    subseq_len = len(subseq)
    matches = []
    for idx in range(input_ids.size(0) - subseq_len + 1):
        if torch.equal(input_ids[idx:idx + subseq_len], subseq):
            return [idx]
    return []

def get_labels(logprobs: torch.Tensor, tokens, size_vocab: int = 151936, offset: int = 0):
    """Get the greedy max probability tokenized sequence from the logprobs."""
    labels = torch.full((len(logprobs), size_vocab), torch.finfo(torch.float16).min, dtype=torch.float16)
    for idx, logprobs_token in enumerate(logprobs):
        for token_id, logprob in zip(tokens[idx], logprobs_token):
            labels[idx][token_id] = logprob
    return labels

In [ ]:
# from torch.utils.data import Dataset
# import jsonlines
# import h5py


# PREFIX_ASSISTANT = [198, 151644, 77091, 198]

# class QwenKLDataset(Dataset):
#     """Dataset for finetuning Qwen model with KL divergence loss."""
#     def __init__(
#             self,
#             path_h5,
#             path_jsonl
#         ):
#         self.entries = []
#         with jsonlines.open(path_jsonl, "r") as reader:
#             with h5py.File(path_h5, "r") as f:
#                 for j, line in tqdm(enumerate(reader)):
#                     inputs = tokenizer.apply_chat_template(line, add_generation_prompt=False, tokenize=True, return_dict=True, return_tensors="pt")
#                     idx_subseq = find_subsequence(inputs["input_ids"][0], subseq = torch.tensor(PREFIX_ASSISTANT))[0]
#                     seq_f = [f[f"seq_{j}"][f"step_{i}"]["token_ids"][0] for i in range(len(f[f"seq_{j}"]))]
#                     try:
#                         assert len(inputs["input_ids"][0][idx_subseq+3:-2]) == len(seq_f)
#                     except AssertionError:
#                         print(f"AssertionError {j}, skipping inconsistent detokenization/tokenization")
#                     tokens = [f[f"seq_{j}"][f"step_{i}"]["token_ids"][:] for i in range(len(f[f"seq_{j}"]))]
#                     logprobs = [f[f"seq_{j}"][f"step_{i}"]["logprobs"][:] for i in range(len(f[f"seq_{j}"]))]
#                     inputs["input_ids"] = inputs["input_ids"].cuda()
#                     self.entries.append(
#                         {
#                             "inputs": inputs,
#                             "idx_subseq": idx_subseq + 3,
#                             "seq_len": len(seq_f),
#                             "logprobs": torch.tensor(np.array(logprobs)),
#                             "tokens": torch.tensor(np.array(tokens))
#                         }
#                     )

#     def __len__(self):
#         return len(self.entries)

#     def __getitem__(self, idx):
#         """
#         Return for a given entry:

#         new_tokens: torch.Tensor, the tokenized conversation with the logprobs of the assistant answer inserted.
#         idx_match: int, the index at which the assistant answer starts in the new_tokens.
#         len_logprob_sequence: int, the tokenized length of the assistant answer.
#         labels: torch.Tensor, the logprobs of the assistant answer for each token.
#         """
#         entry = self.entries[idx]
#         entry["labels"] = get_labels(entry["logprobs"], entry["tokens"], offset=0, size_vocab=151936).cuda()

#         return entry

from torch.utils.data import Dataset
import jsonlines
import h5py
import torch
import numpy as np
from tqdm.auto import tqdm


PREFIX_ASSISTANT = [198, 151644, 77091, 198]


class QwenKLDataset(Dataset):
    """Dataset for finetuning Qwen model with KL divergence loss."""

    def __init__(
        self,
        path_h5,
        path_jsonl
    ):
        self.entries = []

        with jsonlines.open(path_jsonl, "r") as reader:
            with h5py.File(path_h5, "r") as f:

                for j, line in tqdm(enumerate(reader)):

                    inputs = tokenizer.apply_chat_template(
                        line,
                        add_generation_prompt=False,
                        tokenize=True,
                        return_dict=True,
                        return_tensors="pt"
                    )

                    idx_subseq = find_subsequence(
                        inputs["input_ids"][0],
                        subseq=torch.tensor(PREFIX_ASSISTANT)
                    )[0]

                    seq_f = [
                        f[f"seq_{j}"][f"step_{i}"]["token_ids"][0]
                        for i in range(len(f[f"seq_{j}"]))
                    ]

                    try:
                        assert (
                            len(
                                inputs["input_ids"][0][
                                    idx_subseq + 3:-2
                                ]
                            )
                            == len(seq_f)
                        )
                    except AssertionError:
                        print(
                            f"AssertionError {j}, "
                            "skipping inconsistent "
                            "detokenization/tokenization"
                        )
                        continue

                    tokens = [
                        f[f"seq_{j}"][f"step_{i}"]["token_ids"][:]
                        for i in range(len(f[f"seq_{j}"]))
                    ]

                    logprobs = [
                        f[f"seq_{j}"][f"step_{i}"]["logprobs"][:]
                        for i in range(len(f[f"seq_{j}"]))
                    ]

                    # IMPORTANT:
                    # Keep everything on CPU inside the dataset.
                    self.entries.append(
                        {
                            "inputs": inputs,
                            "idx_subseq": idx_subseq + 3,
                            "seq_len": len(seq_f),
                            "logprobs": torch.tensor(
                                np.array(logprobs)
                            ),
                            "tokens": torch.tensor(
                                np.array(tokens)
                            )
                        }
                    )

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]

        # Generate labels temporarily on CPU.
        # DO NOT assign them back into self.entries[idx].
        labels = get_labels(
            entry["logprobs"],
            entry["tokens"],
            offset=0,
            size_vocab=151936
        )

        # Return a new dictionary
        return {
            "inputs": entry["inputs"],
            "idx_subseq": entry["idx_subseq"],
            "seq_len": entry["seq_len"],
            "labels": labels
        }

In [ ]:
print(QwenKLDataset)

<class '__main__.QwenKLDataset'>


KL-Divergence is a metric often used to quantify the difference of probability mass between two distributions.
It's not mathematically defined as a distance because of its asymetric nature:

$$D_{KL}(P \parallel Q) = \sum_{x} P(x) \log \frac{P(x)}{Q(x)}$$

In knowledge distillation, we often minimize the **Kullback–Leibler divergence** between the teacher’s output distribution $P_T$ and the student’s output distribution $P_S$.

$$
D_{KL}(P_T \parallel P_S) = \sum_x P_T(x) \log \frac{P_T(x)}{P_S(x)}
$$
$$
D_{KL}(P_S \parallel P_T) = \sum_x P_S(x) \log \frac{P_S(x)}{P_T(x)}
$$


<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 8:</br>
What qualitative difference would it make if we minimize $D_{KL}(P_T \parallel P_S)$ instead of $D_{KL}(P_S \parallel P_T)$? How would it affect the convergence of the student distribution?

<hr style="border:10px solid red"> </hr>  
</font></h4>



<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 8: </b><br>
Your answer here.

Different behavior as KL Divergence is asymmetric.

With teacher → student:

D_{KL}(P_T - P_S)

the loss strongly penalizes the student when it assigns too little probability to outcomes that the teacher considers likely. The student is therefore encouraged to cover the teacher’s probability distribution.

With student → teacher:

D_{KL}(P_S - P_T)

the loss strongly penalizes the student for assigning probability to outcomes that the teacher considers unlikely. Consequently, the student tends to concentrate probability on the teacher’s strongest modes rather than covering all of them.

Therefore, D_{KL}(P_T - P_S) generally encourages the student to reproduce the teacher’s full distribution more broadly, whereas the reverse KL may converge toward only the teacher’s dominant modes.

<hr style="border:10px solid green"> </hr>
</font></h4>

**You need to ensure you restarted the kernel and have sufficiently free memory (no 7B model running)**

check with `! nvidia-smi`

In [ ]:
! nvidia-smi

Wed Aug 26 09:28:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----




This code is very basic, and allows you to test a training over the small number of samples you've generated. Due to limitations of Google Colab we cannot go much further, but this first part should have given you the basics on how to perform white-box distillation.

<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 4: Complete The Training Code</br>

<hr style="border:10px solid blue"> </hr>
</font></h4>


- Loss function
- gradient accumulation handling



In [ ]:
import os
import glob

print("Current directory:", os.getcwd())
print("HDF5 files:", glob.glob("**/*.h5", recursive=True))
print("JSONL files:", glob.glob("**/*.jsonl", recursive=True))

Current directory: /content
HDF5 files: ['logprobs.h5']
JSONL files: ['questions.jsonl', 'conversations_rag.jsonl']


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

import torch
import numpy as np
import jsonlines
import h5py
import math


tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct"
)

In [ ]:
import math
import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Prefer bfloat16 if supported
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
print("Using dtype:", dtype)

student_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=dtype
).cuda()

dataset = QwenKLDataset(
    path_h5="logprobs.h5",
    path_jsonl="conversations_rag.jsonl"
)

loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=lambda x: x[0]
)

num_epochs = 2
grad_accum_steps = 8

optimizer = torch.optim.AdamW(
    student_model.parameters(),
    lr=1e-5
)

updates_per_epoch = math.ceil(len(loader) / grad_accum_steps)
num_training_steps = num_epochs * updates_per_epoch

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

loss_fn = torch.nn.KLDivLoss(
    reduction="batchmean",
    log_target=True
)

student_model.train()
optimizer.zero_grad()

for epoch in range(num_epochs):
    epoch_loss = 0.0

    for i, batch in enumerate(loader):

        input_ids = batch["inputs"]["input_ids"].cuda()

        outputs = student_model(
            input_ids=input_ids
        )

        answer_start = batch["idx_subseq"]
        answer_length = batch["seq_len"]

        student_logits = outputs.logits[
            0,
            answer_start:answer_start + answer_length,
            :
        ]

        # Compute KL in float32 for stability
        student_logprobs = torch.log_softmax(
            student_logits.float(),
            dim=-1
        )

        teacher_logprobs = batch["labels"].to(
            device=student_logprobs.device,
            dtype=torch.float32
        )

        loss = loss_fn(
            student_logprobs,
            teacher_logprobs
        )

        # Stop immediately if something goes wrong
        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch+1}, batch {i+1}"
            )

        epoch_loss += loss.item()

        batch_loss = loss / grad_accum_steps
        batch_loss.backward()

        if (
            (i + 1) % grad_accum_steps == 0
            or (i + 1) == len(loader)
        ):
            torch.nn.utils.clip_grad_norm_(
                student_model.parameters(),
                max_norm=1.0
            )

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        if (i + 1) % 20 == 0:
            print(
                f"Epoch {epoch+1}, "
                f"batch {i+1}/{len(loader)}, "
                f"loss={loss.item():.4f}"
            )

    avg_loss = epoch_loss / len(loader)

    print(
        f"Epoch {epoch+1}: "
        f"average KL loss = {avg_loss:.4f}"
    )

student_model.save_pretrained("finetuned_model")
tokenizer.save_pretrained("finetuned_model")

Using dtype: torch.bfloat16


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

0it [00:00, ?it/s]

Epoch 1, batch 20/200, loss=0.2821
Epoch 1, batch 40/200, loss=0.1206
Epoch 1, batch 60/200, loss=0.3385
Epoch 1, batch 80/200, loss=0.1051
Epoch 1, batch 100/200, loss=0.4353
Epoch 1, batch 120/200, loss=0.3302
Epoch 1, batch 140/200, loss=0.0304
Epoch 1, batch 160/200, loss=0.4160
Epoch 1, batch 180/200, loss=0.2406
Epoch 1, batch 200/200, loss=0.3229
Epoch 1: average KL loss = 0.3114
Epoch 2, batch 20/200, loss=0.0504
Epoch 2, batch 40/200, loss=0.3786
Epoch 2, batch 60/200, loss=0.4789
Epoch 2, batch 80/200, loss=0.3100
Epoch 2, batch 100/200, loss=0.4894
Epoch 2, batch 120/200, loss=0.3740
Epoch 2, batch 140/200, loss=0.2381
Epoch 2, batch 160/200, loss=0.3850
Epoch 2, batch 180/200, loss=0.3600
Epoch 2, batch 200/200, loss=0.3353
Epoch 2: average KL loss = 0.2505


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('finetuned_model/tokenizer_config.json',
 'finetuned_model/chat_template.jinja',
 'finetuned_model/tokenizer.json')

# Part 2 - Retrieval Augmented Generation (RAG)

In this section, we will discuss the concept of **Retrieval-Augmented Generation (RAG)** — a framework that combines **information retrieval** and **language generation**. RAG enables language models to access **external knowledge sources** at inference time, reducing hallucinations and improving factual accuracy.

We will explore how to:
- Build and index a **Vector database** from a corpus (here: Wikipedia sample).
- Retrieve the most relevant documents given a query using **embedding-based similarity**.
- Integrate retrieval results into the **generation pipeline** to produce context-aware answers.


In [1]:
# restart the session and run
!pip -q install chromadb==0.4.22
!pip -q install "numpy<2.0" --force-reinstall
!pip -q install datasets==2.21.0 pandas==2.2.2

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.0/509.0 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 117.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.6/567.6 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 121.4 MB/s e

In [3]:
%pip install --force-reinstall --no-cache-dir "numpy==2.2.6" "pandas==2.2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 30.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 192.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 168.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 182.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 kB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 97.7 MB/s eta 0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalling pytz-2025.2:
      Successfully uninstalled pytz-2025.2
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2026.3
    Uninstalling tzdata-2026.3:
      Successfully uninstalled tzdata-2026.3
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled si

In [26]:
import os
import json
import random
import torch
import pandas as pd
import torch.nn.functional as F
from tqdm import tqdm
from datasets import load_dataset
from IPython.display import display, HTML
from transformers import AutoTokenizer, AutoModel

In [27]:
# --- Configuration ---
SEED = 42
NUM_ROWS = 1000
DATA_PATH = "wikipedia_20231101_en_1000.csv"

if os.path.exists(DATA_PATH):
    print(f"✅ Found existing dataset at {DATA_PATH}")
    df = pd.read_csv(DATA_PATH)
else:
    print("⏳ Generating new dataset from Wikimedia (English, 2023-11-01)...")
    random.seed(SEED)

    # Load the Wikipedia dataset
    stream_ds = load_dataset(
        "wikimedia/wikipedia",
        "20231101.en",
        split="train",
        streaming=True
    )

    buffered_stream = stream_ds.shuffle(seed=SEED, buffer_size=200_000)

    sampled = []
    for ex in buffered_stream:
        try:
            if int(ex["id"]) % 2 == 0:
                sampled.append(ex)
            if len(sampled) >= NUM_ROWS:
                break
        except:
            continue

    # Create DataFrame
    df = pd.DataFrame(sampled)[["id", "url", "title", "text"]]
    df.to_csv(DATA_PATH, index=False)
    print(f"💾 Dataset saved to {DATA_PATH}")


✅ Found existing dataset at wikipedia_20231101_en_1000.csv


In [28]:
#Display Basic Info ---
print("Sampled shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Sampled shape: (1000, 4)
Columns: ['id', 'url', 'title', 'text']


,id,url,title,text
0,64741026,https://en.wikipedia.org/wiki/Indrani%20Perera,Indrani Perera,Indrani Perera (Sinhala:ඉන්ද්‍රානි පෙරේරා: bor...
1,66847582,https://en.wikipedia.org/wiki/August%20Laur,August Laur,August Laur (9 October 1886 Vana-Põltsamaa Par...
2,66467526,https://en.wikipedia.org/wiki/Daniele%20Solcia,Daniele Solcia,Daniele Solcia (born 7 March 2001) is an Itali...
3,65913988,https://en.wikipedia.org/wiki/Eric%20Takabatake,Eric Takabatake,Eric Takabatake (born 9 January 1991) is a Bra...
4,64723650,https://en.wikipedia.org/wiki/Nafissath%20Radji,Nafissath Radji,Nafissath Radji (born 2 August 2002 in Porto-N...


In [29]:
# Visualize Random Wikipedia Articles ---

NUM_EXAMPLES = 3  # number of random samples to show
samples = df.sample(NUM_EXAMPLES, random_state=random.randint(0, 10000))

for _, row in samples.iterrows():
    display(HTML(f"""
    <hr style="border:2px solid #ccc">
    <h3><b>Title:</b> {row['title']}</h3>
    <p><b>URL:</b> <a href="{row['url']}" target="_blank">{row['url']}</a></p>
    <p style="text-align: justify;"><b>Text:</b><br>{row['text']}</p>
    """))


### **Document Chunking**

The first step in building a RAG pipeline is **chunking**, where large documents are divided into smaller, semantically coherent pieces.  
Chunking allows the retriever to work on manageable text segments instead of entire documents, improving retrieval precision and reducing computational load.  


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 1: Naive Fixed-Length Chunking  
Split each document into overlapping fixed-length chunks to prepare text for retrieval.
<hr style="border:10px solid blue"> </hr>
</font></h4>


In [30]:

def text_splitting(text, chunk_length=300, chunk_overlap=100):
    """
    Splits text into fixed-length chunks with overlap.
    """
    out = []
    stride = chunk_length - chunk_overlap## FILL THE GAP: define the stride as the effective step between chunks
    for i in range(0, len(text), stride):
        chunk = text[i : i+chunk_length] ## FILL THE GAP: extract a substring of size 'chunk_length' starting at 'i'
        out.append(chunk)
    return out

# Apply to all documents
df["naive_chunks"] = df["text"].apply(lambda t: text_splitting(t, chunk_length=300, chunk_overlap=100))

num_chunks = df["naive_chunks"].apply(len)
print(f"Average number of chunks per document: {num_chunks.mean():.2f}")
print(f"Total number of chunks: {num_chunks.sum()}")

example_idx = 0
print("\n--- Example document ---")
print("Title:", df.iloc[example_idx]["title"])
print("Original length:", len(df.iloc[example_idx]["text"]))
print("Number of chunks:", len(df.iloc[example_idx]["naive_chunks"]))
print("\nFirst 2 chunks:\n")
for i, c in enumerate(df.iloc[example_idx]["naive_chunks"][:2]):
    print(f"Chunk {i+1}:\n{c[:400]}\n{'-'*80}")

Average number of chunks per document: 11.50
Total number of chunks: 11498

--- Example document ---
Title: Indrani Perera
Original length: 3012
Number of chunks: 16

First 2 chunks:

Chunk 1:
Indrani Perera (Sinhala:ඉන්ද්‍රානි පෙරේරා: born 15 February), is a Sri Lankan singer and playback singer. Indrani along with Clarence Wijewardena and Annesley Malewana are known as "The Original Sinhala Pop Trio".

Early life 
She was born on 15 February in Borella, and is the second of three girls 
--------------------------------------------------------------------------------
Chunk 2:
la Pop Trio".

Early life 
She was born on 15 February in Borella, and is the second of three girls in the family. Her father, Abeypala Perera was a Buddhist and mother, Muriel Perera was a Christian. She has one elder sister, Mallika and one younger sister, Iranganie. Indrani studied at  Presbyteri
--------------------------------------------------------------------------------


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 2: Paragraph-Aware Chunking  
Implement a smarter chunking strategy by using the ('.') as a boundary to split text into sentences or short paragraphs, then regroup them until reaching the desired chunk length.
<hr style="border:10px solid blue"> </hr>
</font></h4>


In [31]:
def text_splitting_paragraph(text, chunk_length=300):
    """
    Splits text by sentences/paragraphs (using '.' as boundary)
    and groups them until reaching the desired chunk length.
    """
    out = []
    paragraph_list =  text.split('.')## FILL THE GAP: split the text into smaller parts using '.' as a separator
    current_text = ""
    length = 0
    for par in paragraph_list:
        if length != 0 and length + len(par) < chunk_length:
            current_text += " " + par## FILL THE GAP: extend the ongoing chunk with the next segment
            length += len(par) + 1## FILL THE GAP: increment the total length accordingly
        else:
            if len(current_text) != 0:
                out.append(current_text.strip())## FILL THE GAP: store the current completed chunk before starting a new one)
            current_text = par## FILL THE GAP: initialize a new chunk with the current paragraph
            length = len(par)## FILL THE GAP: reset the chunk length counter
    if len(current_text) > 0:
        out.append(current_text.strip())## FILL THE GAP: add the last remaining chunk to the list)
    return out


# Apply to all documents
df["paragraph_chunks"] = df["text"].apply(lambda t: text_splitting_paragraph(t, chunk_length=300))

# Compute stats
num_chunks_par = df["paragraph_chunks"].apply(len)
print(f"Average number of paragraph-based chunks per document: {num_chunks_par.mean():.2f}")
print(f"Total number of paragraph-based chunks: {num_chunks_par.sum()}")

# Example comparison
example_idx = 432 #Check out other examples
print("\n--- Example document ---")
print("Title:", df.iloc[example_idx]['title'])
print("Original length:", len(df.iloc[example_idx]['text']))
print(f"Character based chunks: {len(df.iloc[example_idx]['naive_chunks'])}")
print(f"Paragraph based chunks: {len(df.iloc[example_idx]['paragraph_chunks'])}")

print("\nParagraph chunk preview:\n")
for i, c in enumerate(df.iloc[example_idx]['paragraph_chunks'][:6]):
    print(f"Chunk {i+1}:\n{c[:400]}\n{'-'*80}")


Average number of paragraph-based chunks per document: 8.91
Total number of paragraph-based chunks: 8914

--- Example document ---
Title: John Ballantine (banker)
Original length: 12204
Character based chunks: 62
Paragraph based chunks: 50

Paragraph chunk preview:

Chunk 1:
John Ballantine (1743–1812), was a Scottish merchant and banker and one of the greatest friends, admirers and closest confidants of Robert Burns
--------------------------------------------------------------------------------
Chunk 2:
Significantly Ballantine gave the poet advice on the selection of poems for his First Kilmarnock Edition as well as being asked for his opinion on the bard's poems 

Life and character
John was born in Ayr to William Ballantine, a baillie in Ayr and his mother was Elizabeth Bowman
--------------------------------------------------------------------------------
Chunk 3:
He was a merchant and a Banker and in 1787 he became the Provost of Ayr, during which time he helped establish Ayr Ac

<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 1:  
In the paragraph-aware chunking method above, we simply split Wikipedia text using the '.' delimiter to approximate sentence boundaries.  
Discuss whether this is an effective strategy for creating meaningful chunks in a RAG system.
Propose one or more improved chunking strategies that could better capture document structure — and you may include code snippets to justify or demonstrate your approach.  
<hr style="border:10px solid red"> </hr>  
</font></h4>


<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 1: </b><br>
Your answer here.

Splitting Wikipedia text only on "." is a simple approximation, but it is not fully reliable for meaningful RAG chunks. A period does not always mean the end of a sentence: it can appear in abbreviations like "Dr.", decimal numbers like "3.14", URLs, or initials. It can also ignore larger document structure such as headings and paragraphs. As a result, some chunks may still be semantically awkward or may combine unrelated ideas.

A better approach would be to use a proper sentence tokenizer or a structure-aware splitter, then group complete sentences until a target token or character length is reached. We could also preserve headings/paragraph boundaries, or use semantic chunking where neighboring sentences are grouped based on embedding similarity. This would produce chunks that are more coherent and therefore more useful for retrieval.

<hr style="border:10px solid green"> </hr>
</font></h4>

<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 2: Consider a scenario where you want to perform RAG on source code (e.g., Python files, Java classes) instead of natural language text. Would the chunking methods demonstrated above (character-based and sentence/paragraph based with boundaries) work effectively for code? Explain why or why not, and describe how you would approach chunking source code to maintain semantic coherence. What specific characteristics of code structure would you need to consider?

<hr style="border:10px solid red"> </hr>
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 2: </b><br>
Your answer here.

The same chunking methods are not ideal for source code. Character-based chunking can split a function or class in the middle, and sentence/paragraph-based splitting does not make sense because code is structured by syntax rather than natural-language punctuation.

For source code, chunking should respect semantic units such as functions, methods, classes, modules, or logical blocks. Ideally, we would parse the code with an AST or language-specific parser and create chunks around these structures. Important characteristics include function and class boundaries, indentation or braces, imports, comments/docstrings, variable scope, and dependencies between methods or functions. This preserves semantic coherence and makes retrieved code chunks much more useful for answering programming questions.

<hr style="border:10px solid green"> </hr>
</font></h4>

In [32]:
# Saving chunks ---

# Flatten chunks into a new DataFrame
records = []
for _, row in df.iterrows():
    doc_id = row["id"]
    title = row["title"]
    url = row["url"]
    for i, chunk in enumerate(row["paragraph_chunks"]):
        records.append({
            "doc_id": doc_id,
            "title": title,
            "url": url,
            "chunk_id": f"{doc_id}_chunk_{i}",
            "chunk_text": chunk.strip()
        })

# Create the flattened chunks DataFrame
chunks_df = pd.DataFrame(records)
print(f"Total chunks: {len(chunks_df)}")
print(f"Average chunk length: {chunks_df['chunk_text'].apply(len).mean():.2f} characters\n")

# Show example
print("Example rows:")
display(chunks_df.head())

chunks_df.to_csv("wikipedia_chunks.csv", index=False)
print("Chunks saved to 'wikipedia_chunks.csv'")
print(f"Total chunks: {len(chunks_df)}")


Total chunks: 8914
Average chunk length: 244.80 characters

Example rows:


,doc_id,title,url,chunk_id,chunk_text
0,64741026,Indrani Perera,https://en.wikipedia.org/wiki/Indrani%20Perera,64741026_chunk_0,Indrani Perera (Sinhala:ඉන්ද්‍රානි පෙරේරා: bor...
1,64741026,Indrani Perera,https://en.wikipedia.org/wiki/Indrani%20Perera,64741026_chunk_1,Early life \nShe was born on 15 February in Bo...
2,64741026,Indrani Perera,https://en.wikipedia.org/wiki/Indrani%20Perera,64741026_chunk_2,Indrani studied at Presbyterian Girls School ...
3,64741026,Indrani Perera,https://en.wikipedia.org/wiki/Indrani%20Perera,64741026_chunk_3,She met him during the production of his song ...
4,64741026,Indrani Perera,https://en.wikipedia.org/wiki/Indrani%20Perera,64741026_chunk_4,After that he studied A/L at the Royal Institu...


Chunks saved to 'wikipedia_chunks.csv'
Total chunks: 8914


## <b>Part II: Embedding</b>

After chunking our documents, the next step is to convert text chunks into vector representations (embeddings). These embeddings capture the semantic meaning of the text in a high-dimensional space, allowing us to measure similarity between chunks and queries mathematically.

We will use **sentence-transformers/all-MiniLM-L6-v2**, a compact and efficient embedding model that produces 384-dimensional embeddings for English text. This model offers a strong balance between performance and computational efficiency, making it well-suited for our RAG pipeline.


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 3: </b><br>
Fill in the <code>embed()</code> function to encode text chunks and generate normalized embeddings using <code>sentence-transformers/all-MiniLM-L6-v2</code>.  
Then, apply it to all documents in <code>chunks_df</code> and store the results.
<hr style="border:10px solid blue"> </hr>
</font></h4>


In [33]:
# --- 2.3 Embedding Generation ---
# Load model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print(f"Loaded embedding model: {model_name}")


# # --- Define embedding function ---
# def embed(text_list, doc_type="document"):
#     """
#     Encodes a list of texts and returns normalized embeddings.
#     """
#     encoded = tokenizer(
#         [f"search_{doc_type}: {t}" for t in text_list],
#         padding=True,
#         truncation=True,
#         return_tensors="pt"
#     ).to(device)

#     with torch.no_grad():
#         output = model(encoded) ## FILL THE GAP: forward pass through the model to obtain hidden states
#         token_embeddings = output.last_hidden_state ## FILL THE GAP: extract the last hidden state from the output
#         pooled = token_embeddings.sum() ## FILL THE GAP: aggregate token embeddings (e.g., by summing along sequence dimension)
#         pooled = F.normalize(
#             pooled,
#             p=2,
#             dim=1
#         )## FILL THE GAP: apply L2 normalization along the embedding dimension
#     return pooled.cpu()
def embed(text_list, doc_type="document"):
    """
    Encodes a list of texts and returns normalized embeddings.
    """

    # Tokenize
    encoded = tokenizer(
        [f"search_{doc_type}: {t}" for t in text_list],
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    # Move individual tensors to GPU/CPU
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():

        # Forward pass
        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # [batch, sequence_length, 384]
        token_embeddings = output.last_hidden_state

        # Mask padding tokens
        mask = attention_mask.unsqueeze(-1).expand(
            token_embeddings.size()
        ).float()

        # Sum only real token embeddings
        summed = torch.sum(
            token_embeddings * mask,
            dim=1
        )

        # Number of real tokens
        counts = torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

        # Mean pooling
        pooled = summed / counts

        # L2 normalization
        pooled = F.normalize(
            pooled,
            p=2,
            dim=1
        )

    return pooled.cpu()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


In [34]:
# --- Test with one example ---
sample_text = ["Artificial intelligence is transforming the world."]
sample_emb = embed(sample_text)
print(f"Sample embedding shape: {sample_emb.shape}")

# --- Apply to all chunks ---
print(f"\nGenerating embeddings for {len(chunks_df)} chunks...")

emb_list = []
for i in tqdm(range(0, len(chunks_df), 32)):
    batch = chunks_df["chunk_text"].iloc[i:i+32].tolist()
    emb = embed(batch, doc_type="document")
    emb_list.append(emb)

chunk_embeddings = torch.cat(emb_list, dim=0).numpy()
chunks_df["embedding"] = list(chunk_embeddings)

print("\nEmbeddings generated and added to DataFrame.")
print(chunks_df[["chunk_id", "title", "embedding"]].head())

Sample embedding shape: torch.Size([1, 384])

Generating embeddings for 8914 chunks...


100%|██████████| 279/279 [00:15<00:00, 18.51it/s]


Embeddings generated and added to DataFrame.
           chunk_id           title  \
0  64741026_chunk_0  Indrani Perera   
1  64741026_chunk_1  Indrani Perera   
2  64741026_chunk_2  Indrani Perera   
3  64741026_chunk_3  Indrani Perera   
4  64741026_chunk_4  Indrani Perera   

                                           embedding  
0  [-0.10768968, -0.019213079, -0.07229619, -0.06...  
1  [-0.013653568, 0.0043238313, -0.11352289, 0.08...  
2  [-0.07606235, -0.07166081, -0.014079438, -0.00...  
3  [-0.10031879, 0.057519175, -0.05016728, 0.0566...  
4  [-0.056192856, 0.0033603979, -0.007968712, 0.0...  


<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 3: In most retrieval systems, embeddings are represented as fixed-size vectors. Can you think of a way to design embeddings that can flexibly adjust their size or level of detail while still preserving meaningful similarity between representations? How could such an approach benefit Retrieval-Augmented Generation (RAG) systems in practice, particularly for improving efficiency or adapting to different computational budgets?

<hr style="border:10px solid red"> </hr>
<i>Hint:</i> You can find the answer in the paper <a href="https://arxiv.org/pdf/2205.13147" target="_blank">Matryoshka Representation Learning</a>.
</font></h4>


Matryoshka Representation Learning trains a single embedding so that its first few dimensions already form a useful representation, while adding later dimensions progressively provides more detail. For example, the same model can produce meaningful embeddings using the first 64, 128, 256, or all 768 dimensions. The smaller embedding is therefore a prefix of the larger one rather than a separately trained representation.

In a RAG system, short embeddings can be used for fast, memory-efficient retrieval over a large collection, while longer embeddings can be used when higher accuracy is required. A practical approach is to retrieve candidates using low-dimensional embeddings and then rerank them using more dimensions. This reduces storage, memory usage, and similarity-computation costs while allowing the system to adapt to different latency and computational budgets.


### **Building a Simple Vector Database**

After generating embeddings for all our document chunks, the next step is to **store** them in a structure that allows fast similarity search.  
In this section, we will build a **simple in-memory vector database** using PyTorch tensors.  
Each entry in the database will correspond to a text chunk and its embedding, enabling efficient retrieval based on vector similarity.


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 4: </b><br>
Fill in the code to populate the database by computing embeddings for all text chunks in <code>chunks_df</code> using the <code>embed()</code> function.
<hr style="border:10px solid blue"> </hr>
</font></h4>


In [35]:

def populate_database(chunks_df, batch_size=16):
    """
    Populates a vector database from precomputed chunks_df.
    """
    n_chunks = len(chunks_df)

    sample_emb = embed([chunks_df["chunk_text"].iloc[0]], doc_type="document")## FILL THE GAP: compute one sample embedding to infer the output dimension
    output_dim = sample_emb.shape[1]## FILL THE GAP: extract the embedding dimension from the sample

    vectorial_database = torch.empty(
        (n_chunks, output_dim),
        dtype=sample_emb.dtype
    )## FILL THE GAP: initialize an empty tensor to store all embeddings
    chunk_list = chunks_df["chunk_text"].tolist()

    print(f"Populating vector database with {n_chunks} chunks...")

    n = 0
    for i in range(0, n_chunks, batch_size):
        batch = chunk_list[i:i+batch_size]## FILL THE GAP: select a batch of chunk texts
        embeddings = embed(batch, doc_type="document") ## FILL THE GAP: compute embeddings for the current batch
        vectorial_database[n:n + len(batch)] = embeddings## FILL THE GAP: store embeddings in the tensor
        n += len(batch)

    return chunk_list, vectorial_database

In [36]:
#Build Vector Database
chunk_list, vectorial_database = populate_database(chunks_df)

print("\n✅ Vector database successfully built.")
print(f"Total stored chunks: {len(chunk_list)}")
print(f"Database tensor shape: {tuple(vectorial_database.shape)}")

Populating vector database with 8914 chunks...

✅ Vector database successfully built.
Total stored chunks: 8914
Database tensor shape: (8914, 384)


In [37]:
#Save vector databse
os.makedirs("vector_db", exist_ok=True)

# Save tensor + chunk list
torch.save(vectorial_database, "vector_db/vectorial_database.pth")

with open("vector_db/chunk_list.json", "w", encoding="utf-8") as f:
    json.dump(chunk_list, f, indent=4, ensure_ascii=False)

print("✅ Saved:")
print(" - vector_db/vectorial_database.pth")
print(" - vector_db/chunk_list.json")

✅ Saved:
 - vector_db/vectorial_database.pth
 - vector_db/chunk_list.json


In [38]:
# Load the database
vectorial_database = torch.load("vector_db/vectorial_database.pth", map_location=device)
vectorial_database.requires_grad_(False)

with open("vector_db/chunk_list.json", "r", encoding="utf-8") as f:
    chunk_list = json.load(f)

print(f"✅ Loaded {len(chunk_list)} chunks.")
print(f"Database shape: {vectorial_database.shape}\n")

# Inspect first few entries
for i, embedding_vector in enumerate(vectorial_database[:5]):
    print(f"Vector {i} → {embedding_vector[:5]}")
    print(f"Text snippet: {chunk_list[i][:300]}\n")


✅ Loaded 8914 chunks.
Database shape: torch.Size([8914, 384])

Vector 0 → tensor([-0.1077, -0.0192, -0.0723, -0.0637, -0.0328], device='cuda:0')
Text snippet: Indrani Perera (Sinhala:ඉන්ද්‍රානි පෙරේරා: born 15 February), is a Sri Lankan singer and playback singer  Indrani along with Clarence Wijewardena and Annesley Malewana are known as "The Original Sinhala Pop Trio"

Vector 1 → tensor([-0.0137,  0.0043, -0.1135,  0.0896, -0.0390], device='cuda:0')
Text snippet: Early life 
She was born on 15 February in Borella, and is the second of three girls in the family  Her father, Abeypala Perera was a Buddhist and mother, Muriel Perera was a Christian  She has one elder sister, Mallika and one younger sister, Iranganie

Vector 2 → tensor([-0.0761, -0.0717, -0.0141, -0.0099, -0.0980], device='cuda:0')
Text snippet: Indrani studied at  Presbyterian Girls School in Regent Street  She studied Kandyan Dancing in the school  Her sister Mallika has been singing since 1965 where she was the playback

#### **Defining Similarity Metrics for Retrieval**

After populating our vector database with embeddings, the next step in a RAG pipeline is to define a *similarity metric* to measure how close two vectors are in the embedding space.  
Common metrics include **dot product**, **L2 distance**, and **cosine similarity**.  

In most retrieval systems, cosine similarity is preferred because it measures the *angle* between two vectors rather than their magnitude, allowing comparison based purely on semantic direction instead of scale.


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 5: </b><br>
Fill in the code to implement the <code>cosine_similarity()</code> function.  
<hr style="border:10px solid blue"> </hr>
</font></h4>


In [39]:
def cosine_similarity(query_embeddings, doc_embeddings):
    """
    Computes cosine similarity between query and document embeddings
    using manual normalization.
    """
    query_magnitudes = torch.linalg.vector_norm(query_embeddings, ord=2, dim=1, keepdim=True) ## FILL THE GAP: calculate vector length for each query
    normalized_queries = query_embeddings / query_magnitudes ## FILL THE GAP: normalize queries using their magnitudes

    doc_magnitudes = torch.linalg.norm(doc_embeddings, ord=2, dim=1, keepdim=True) ## FILL THE GAP: calculate vector length for each document
    normalized_docs = doc_embeddings / doc_magnitudes ## FILL THE GAP: normalize documents using their magnitudes

    similarity_matrix = normalized_queries @ normalized_docs.T ## FILL THE GAP: perform dot product between normalized queries and documents

    return similarity_matrix


# --- Example test ---
query_embeddings = embed([
    "What is t-SNE?",
    "Who is Laurens van der Maaten?"
], "query")

doc_embeddings = embed([
    "t-SNE is a dimensionality reduction algorithm created by Laurens van der Maaten."
], "document")

with torch.no_grad():
    sim_cos = cosine_similarity(query_embeddings, doc_embeddings)

print("🔍 Example cosine similarity scores:\n", sim_cos)

🔍 Example cosine similarity scores:
 tensor([[0.6621],
        [0.4333]])


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 6: </b><br>
Fill in the code to complete the <code>retrieve()</code> function.  
It should encode the input query using <code>embed()</code>, compute similarity with all vectors in <code>vectorial_database</code>, and return the top-<i>k</i> most similar text chunks.
<hr style="border:10px solid blue"> </hr>
</font></h4>


In [40]:
def retrieve(query,
             vectorial_database=vectorial_database,
             chunk_list=chunk_list,
             topk=5,
             verbose=False):
    """
    Retrieves top-k most similar chunks to a query using dot-product similarity.
    """
    with torch.no_grad():
        query_embedding = embed([query], doc_type='query') ## FILL THE GAP: encode the input query using the embed() function
        query_embedding = query_embedding.to(vectorial_database.device) ## FILL THE GAP: move the query embedding to the correct device
        similarity_scores = cosine_similarity(query_embedding, vectorial_database)## FILL THE GAP: compute similarity between query and database embeddings
        effective_topk = min(topk, len(chunk_list))

        topk_results = torch.topk(
            similarity_scores,
            k=effective_topk,
            dim=1
        )## FILL THE GAP: extract the top-k highest similarity scores and indices

        if verbose:
            for score, idx in zip(topk_results.values[0], topk_results.indices[0]):
                print(f"\nScore: {score:.4f}")
                print(f"Chunk:\n{chunk_list[idx][:500]}\n{'-'*80}")

        retrieved_chunks = [
            chunk_list[idx.item()]
            for idx in topk_results.indices[0]
        ] ## FILL THE GAP: select text chunks corresponding to the top-k indices
        return "\n\n".join(retrieved_chunks) ## FILL THE GAP: return concatenated retrieved chunks as a single string


In [41]:
# Example query
query = "When was Luigi Boria born?" #Try different queries based on the documents in the wikipedia dataset
result = retrieve(query, topk=3, verbose=True)



Score: 0.5807
Chunk:
Luigi Boria (born April 23, 1958) is a Venezuelan-born American politician who served as mayor of Doral, Florida from 2012 to 2016  Boria defeated former Miami-Dade School Board member Frank Bolaños in the 2012 elections, obtaining 54% of the vote
--------------------------------------------------------------------------------

Score: 0.5610
Chunk:
(1675–1684)
 Sede vacante (March 1684–August 1686)
 Nicolò Caranza (1686–1697)
 Giulio Della Rosa (1698–1699)
 Alessandro Roncoveri (1700–1711)
 Adriano Sermattei (1713–1719)
 Gherardo Zandemaria (1719–1731)
 Severino Antonio Missini (1732–1753)
 Girolamo Bajardi (1753–1775)
 Alessandro Garimberti (1776–1813)
Sede vacante (1813–1817)
 Aloisio San Vitale (1817–1836)
 Giovanni Tommaso Neuschel (1836–1843)
 Pier Grisologo Basetti (1843–1857)
Sede vacante (16 June 1857 – 20 June 1859)
 Francesco Ben
--------------------------------------------------------------------------------

Score: 0.5256
Chunk:
Biography 
Boria, who i

<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr> Question 4: There are retrieval methods like BM25 that rely on lexical overlap between the query and documents, and others based on dense embeddings that capture semantic similarity beyond exact word matches. Explain how these two approaches differ in how they represent and compare text. Then, discuss how a hybrid retrieval strategy combining both can overcome their respective limitations and improve retrieval performance in RAG systems. <hr style="border:10px solid red"> </hr> </font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 4: </b><br>
Your answer here.

BM25 represents queries and documents using sparse lexical features based on term frequency, inverse document frequency, and document length. It scores documents according to exact or near-exact word overlap. It is therefore effective for names, technical terms, identifiers, and rare keywords, but it may miss relevant passages expressed using different vocabulary.

Dense retrieval encodes queries and documents as continuous vectors and compares them using a similarity measure such as cosine similarity. Because the embeddings capture semantic meaning, dense retrieval can match paraphrases and conceptually related passages even without exact word overlap. However, it may overlook important exact terms or retrieve passages that are semantically related but do not contain the required information.

A hybrid system combines BM25 and dense-retrieval scores or merges their candidate lists, for example using Reciprocal Rank Fusion. It therefore preserves BM25’s exact keyword matching while benefiting from the semantic generalization of dense embeddings. The combined candidates can subsequently be processed by a reranker, generally improving both recall and precision.

<hr style="border:10px solid green"> </hr>
</font></h4>


In real-world RAG systems, instead of manually storing and comparing vectors, we rely on **vector databases** such as **ChromaDB**, which are optimized for efficient **similarity search**, **indexing**, and **retrieval** at scale.  

These databases provide:
- Fast nearest-neighbor search (e.g., using HNSW graphs)  
- Persistent storage for millions of embeddings  
- Built-in support for different similarity metrics (cosine, L2, inner product)  


In [19]:
# %pip install --upgrade --no-cache-dir chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 144.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 156.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 186.3 MB/s eta 0:00:00
  Attempting uninstall: chromadb
    Found existing installation: chromadb 0.4.22
    Uninstalling chromadb-0.4.22:
      Successfully uninstalled chromadb-0.4.22


In [42]:
import chromadb
from chromadb.config import Settings

# --- Initialize ChromaDB client ---
chroma_client = chromadb.Client(Settings(
    anonymized_telemetry=False,
    allow_reset=True
))

# Reset ensures a clean state
chroma_client.reset()

# --- Create a collection ---
# You can choose the similarity metric: "cosine", "l2", or "ip" (inner product)
collection_name = "wikipedia_chunks"
collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}  # cosine similarity works best for normalized embeddings
)

print(f"✅ Created collection: {collection_name}")


✅ Created collection: wikipedia_chunks


In [44]:
# # Prepare the data
# ids = chunks_df["chunk_id"].tolist()
# embeddings = chunks_df["embedding"].tolist()
# documents = chunks_df["chunk_text"].tolist()

# # Ensure all embeddings are plain Python lists
# embeddings = [e.tolist() if hasattr(e, "tolist") else e for e in embeddings]

# # Add useful metadata for inspection
# metadatas = [
#     {
#         "doc_id": row["doc_id"],
#         "title": row["title"],
#         "url": row["url"]
#     }
#     for _, row in chunks_df.iterrows()
# ]

# # Add to the collection
# collection.add(
#     ids=ids,
#     embeddings=embeddings,
#     documents=documents,
#     metadatas=metadatas
# )

# print(f"\n✅ Added {collection.count()} chunks to the collection")
# print(f"Collection metadata: {collection.metadata}")
ids = [f"chunk_{i}" for i in range(len(chunk_list))]
documents = chunk_list
embeddings = vectorial_database.cpu().numpy().tolist()

# Include this only if your notebook created metadata
metadatas = [
    {"chunk_index": i}
    for i in range(len(chunk_list))
]

batch_size = 1000

for start in range(0, len(ids), batch_size):
    end = min(start + batch_size, len(ids))

    collection.add(
        ids=ids[start:end],
        embeddings=embeddings[start:end],
        documents=documents[start:end],
        metadatas=metadatas[start:end]
    )

    print(f"Added {end}/{len(ids)} chunks")

print("\nNumber of chunks in ChromaDB:", collection.count())

Added 1000/8914 chunks
Added 2000/8914 chunks
Added 3000/8914 chunks
Added 4000/8914 chunks
Added 5000/8914 chunks
Added 6000/8914 chunks
Added 7000/8914 chunks
Added 8000/8914 chunks
Added 8914/8914 chunks

Number of chunks in ChromaDB: 8914


In [45]:
# Encode the query using our embed() function
query = "When was Luigi Boria born?"
query_embedding = embed([query], doc_type="query")[0].tolist()  # get single vector as list

# Query the collection
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3 # number of retrieved results
)

# Display results
print(f"🔎 Query: {query}\n" + "=" * 80)
for i in range(len(results["documents"][0])):
    print(f"\nResult {i+1}:")
    print(f"Title: {results['metadatas'][0][i]['title']}")
    print(f"Similarity score: {1 - results['distances'][0][i]:.4f}")  # cosine distance → similarity
    print(f"Text: {results['documents'][0][i][:300]}...")
    print("-" * 80)


🔎 Query: When was Luigi Boria born?

Result 1:


KeyError: 'title'

<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 5: </b><br>
ChromaDB and other vector databases often rely on Hierarchical Navigable Small World (HNSW) graphs to perform efficient approximate nearest neighbor search.  
Explain how the HNSW algorithm organizes data to enable fast and accurate retrieval in high-dimensional spaces.  
Why is this structure particularly effective for large-scale embedding collections compared to brute-force search?

<hr style="border:10px solid red"> </hr>
<i>Reference:</i> <a href="https://arxiv.org/pdf/1603.09320" target="_blank">Efficient and Robust Approximate Nearest Neighbor Search using Hierarchical Navigable Small World Graphs</a>
</font></h4>


<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 5: </b><br>
HNSW organizes embeddings into a multilayer proximity graph. The upper layers contain relatively few, long-range connections that allow the search to move rapidly across the embedding space. Lower layers contain progressively more points and shorter-range connections, with the bottom layer containing the complete collection.
A query starts at an upper layer and greedily moves toward increasingly similar nodes. It then descends through the layers and performs a more detailed search around the most promising region. This resembles navigating from highways to local roads.
Unlike brute-force retrieval, which compares the query with every stored embedding, HNSW examines only a small subset of promising candidates. It consequently provides fast approximate nearest-neighbor retrieval with high recall and scales effectively to millions of high-dimensional embeddings. Its main trade-offs are additional graph memory and the possibility of missing the exact nearest neighbor.

<hr style="border:10px solid green"> </hr>
</font></h4>


### <b> Two-Stage Retrieval: Dense Retrieval + Reranker</b>

In RAG, the first retrieval step often returns passages that are similar in meaning but not always the most relevant.  
A **reranker** fixes this by re-evaluating the top retrieved chunks using a stronger model that jointly reads the query and each document to assign a more accurate relevance score.


In [46]:
from sentence_transformers import CrossEncoder

# Load reranker model
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL_NAME)

def retrieve_with_reranker(query, collection, initial_k=5, final_k=3):
    """
    Retrieves and reranks candidate documents for a given query.
    Returns both the initial dense results and reranked results.
    """
    query_embedding = embed([query], doc_type="query")[0].tolist()
    candidates = collection.query(query_embeddings=[query_embedding], n_results=initial_k)

    docs = candidates["documents"][0]
    metas = candidates["metadatas"][0]
    dense_scores = [(1 - s) for s in candidates["distances"][0]]

    pairs = [(query, d) for d in docs]
    ce_scores = reranker.predict(pairs)

    reranked = [
        {
            "title": metas[i].get("title", ""),
            "url": metas[i].get("url", ""),
            "text": docs[i],
            "dense_score": dense_scores[i],
            "rerank_score": float(ce_scores[i]),
        }
        for i in range(len(docs))
    ]
    reranked.sort(key=lambda x: x["rerank_score"], reverse=True)

    return docs, metas, dense_scores, reranked[:final_k]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [47]:
# Example usage
query = "Where did Luigi Boria study?" #Try other queries too
docs, metas, dense_scores, top_reranked = retrieve_with_reranker(
    query=query,
    collection=collection,
    initial_k=5,
    final_k=3
)

# --- Display initial dense retrieval ---
print(f"\nInitial dense retrieval (Top 5):\n" + "=" * 80)
for i, (d, s, m) in enumerate(zip(docs, dense_scores, metas), 1):
    print(f"{i}. {m.get('title', '')}  |  Dense similarity: {s:.4f}")
    print(f"Text: {d[:300].replace('\n', ' ')}")
    print("-" * 80)

# --- Display top reranked results ---
print(f"\nAfter Cross-Encoder Reranking (Top 3):\n" + "=" * 80)
for i, item in enumerate(top_reranked, 1):
    print(f"{i}. {item['title']}")
    print(f"Dense similarity: {item['dense_score']:.4f} | Reranker score: {item['rerank_score']:.4f}")
    print(f"Text: {item['text'][:300].replace('\n', ' ')}")
    print("-" * 80)


Initial dense retrieval (Top 5):
1.   |  Dense similarity: 0.5195
Text: Luigi Boria (born April 23, 1958) is a Venezuelan-born American politician who served as mayor of Doral, Florida from 2012 to 2016  Boria defeated former Miami-Dade School Board member Frank Bolaños in the 2012 elections, obtaining 54% of the vote
--------------------------------------------------------------------------------
2.   |  Dense similarity: 0.5106
Text: Biography  Boria, who is also an evangelical Christian pastor, was born in Caracas in 1958 to Italian parents who emigrated to that country  He studied accounting at the Andrés Bello Catholic University, a private institution, in 1982
--------------------------------------------------------------------------------
3.   |  Dense similarity: 0.4701
Text: He is "best known for his collections and floristic studies in the Mexican state of Chiapas, and his ethnobotanical work in that state with various collaborators "  Education and career After graduating f

![Bi-Encoder vs Cross-Encoder Architecture](https://raw.githubusercontent.com/UKPLab/sentence-transformers/master/docs/img/Bi_vs_Cross-Encoder.png)

<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 6:  
The figure above compares a Bi-Encoder and a Cross-Encoder architecture.  
Rerankers such as <code>cross-encoder/ms-marco-MiniLM-L-6-v2</code> use the second approach, jointly encoding the query and document through a single transformer.  
Why does this joint encoding typically yield higher retrieval precision, and why is it applied as a second-stage reranker instead of being used directly for large-scale retrieval?
<hr style="border:10px solid red"> </hr>
</font></h4>


<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 6: </b><br>
A cross-encoder processes the query and document together, allowing self-attention to model direct token-level interactions between them. It can determine whether particular statements in the document answer specific parts of the query, producing more precise relevance scores than a bi-encoder, which embeds the query and document independently before comparing their vectors.
However, cross-encoder document representations cannot be computed and indexed in advance because the document must be jointly processed with each new query. Applying it to every document would require one transformer forward pass per query-document pair and would be prohibitively expensive for a large collection.
It is therefore used as a second-stage reranker: an efficient method such as BM25, dense retrieval, or hybrid retrieval first selects a small candidate set, and the cross-encoder then reranks only those candidates with greater precision.<hr style="border:10px solid green"> </hr>
</font></h4>


### **Integrating Retrieved Context into the LLM’s Prompt**

Now that we can retrieve and rerank the most relevant document chunks,  
we integrate them directly into the **language model’s prompt**.  
This step allows the model to **ground its answer on factual context** rather than relying solely on internal knowledge —  
thereby improving accuracy and reducing hallucinations.


In [48]:
print("Number of chunks in ChromaDB:", collection.count())

Number of chunks in ChromaDB: 8914


In [49]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

gen_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
gen_tok = AutoTokenizer.from_pretrained(gen_model_name, trust_remote_code=True)
gen_model = AutoModelForCausalLM.from_pretrained(
    gen_model_name,
    device_map="auto",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)

def generate(msg, max_new_tokens=128, temperature=0.2):
    messages = [{"role": "user", "content": msg}]
    inputs = gen_tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(gen_model.device)

    outputs = gen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        eos_token_id=gen_tok.eos_token_id,
        pad_token_id=gen_tok.pad_token_id if gen_tok.pad_token_id is not None else gen_tok.eos_token_id
    )
    input_length = inputs["input_ids"].shape[-1]
    return gen_tok.decode(outputs[0, input_length:], skip_special_tokens=True).strip()


# ============================================================
# NO-RAG vs WITH RAG
# ============================================================
query = "When was Luigi Boria born?" #Try other queries

print("ORIGINAL PROMPT\n" + "=" * 60)
print(query)

print("\nANSWER WITHOUT RAG\n" + "=" * 60)
print(generate(query))

docs, metas, dense_scores, top_reranked = retrieve_with_reranker(
    query=query,
    collection=collection,
    initial_k=5,
    final_k=3
)

context = "\n\n".join(h["text"] for h in top_reranked)[:1600]
rag_prompt = (
    f"Use only the context to answer. If unknown, say you don't know.\n\n"
    f"Context:\n{context}\n\n"
    f"Question: {query}\nAnswer:"
)

print("\nAUGMENTED PROMPT\n" + "=" * 60)
print(rag_prompt)

print("\nANSWER WITH RAG\n" + "=" * 60)
print(generate(rag_prompt))


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

ORIGINAL PROMPT
When was Luigi Boria born?

ANSWER WITHOUT RAG
Luigi Boria was born on October 19, 1968. He is an Italian-American actor and director known for his work in both film and television.

AUGMENTED PROMPT
Use only the context to answer. If unknown, say you don't know.

Context:
Luigi Boria (born April 23, 1958) is a Venezuelan-born American politician who served as mayor of Doral, Florida from 2012 to 2016  Boria defeated former Miami-Dade School Board member Frank Bolaños in the 2012 elections, obtaining 54% of the vote

Biography 
Boria, who is also an evangelical Christian pastor, was born in Caracas in 1958 to Italian parents who emigrated to that country  He studied accounting at the Andrés Bello Catholic University, a private institution, in 1982

(1675–1684)
 Sede vacante (March 1684–August 1686)
 Nicolò Caranza (1686–1697)
 Giulio Della Rosa (1698–1699)
 Alessandro Roncoveri (1700–1711)
 Adriano Sermattei (1713–1719)
 Gherardo Zandemaria (1719–1731)
 Severino Antonio

<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr> Question 7: In the RAG prompt construction step, we simply concatenate the retrieved chunks before the question. Discuss potential issues with this naive approach, such as token limits, redundancy, or irrelevant context dilution. Then, explain how we could select, weight, or summarize the retrieved chunks before injecting them into the prompt to improve generation quality and efficiency. <hr style="border:10px solid red"> </hr> </font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 7: </b><br>
Simply concatenating all retrieved chunks can exceed the model’s context limit, increase inference time and cost, and truncate useful information. Retrieved chunks may also be redundant, contradictory, or only weakly relevant. Too much irrelevant text can dilute the useful evidence and make the model more likely to produce an incorrect or unsupported answer.
The context can be improved by first reranking chunks with a cross-encoder and retaining only those above a relevance threshold. Redundancy can be reduced through similarity-based deduplication or diversity-aware methods such as Maximum Marginal Relevance. Chunks may also be weighted according to relevance, reordered so the strongest evidence appears in prominent positions, or compressed to retain only query-relevant sentences.
For long documents, the system can summarize related chunks or use hierarchical retrieval: first select relevant documents, then sections, and finally individual passages. These methods produce a shorter, more focused context and generally improve both generation quality and computational efficiency.
<hr style="border:10px solid green"> </hr>
</font></h4>


#### **To go further**

- Experiment with other chunking methods (e.g., semantic or recursive chunking).  
- Explore **LangChain** and **LlamaIndex** for building modular RAG pipelines.  
- Try **hybrid retrieval** combining sparse (BM25) and dense embeddings.  
- Explore more advanced RAG methods such as **RAG-Fusion**, **Self-RAG**, and **Active-RAG**.  
- Experiment with **Matryoshka Representation Learning** for scalable embeddings.  
- Try **fine-tuning rerankers** or **retrievers** for domain-specific data.  
